## NTS0502b-trip_purpose_by_start_time (Monday to Friday only) England, 2002 onwards
## **When** do peoples start travelling?

### Rationale for Selecting NTS0502b

The National Travel Survey (NTS) provides two complementary temporal datasets:

- **NTS0502a:** *Trip purpose by trip start time*, representing the distribution of trip purposes within each departure time interval.
- **NTS0502b:** *Trip start time by trip purpose*, representing the distribution of departure times for each trip purpose.

This framework adopts **NTS0502b** because the Temporal Layer operates **after** the Mobility Layer within the proposed behavioural generation framework.

The Mobility Layer first assigns a trip purpose to each synthetic individual based on demographic-specific mobility probabilities. Once the trip purpose has been determined, the Temporal Layer generates a plausible trip start time conditioned on that purpose. Therefore, the required probability distribution is:

P(Start Time|Trip Purpose)

rather than


P(Trip Purpose|Start Time|)

This preserves the sequential behavioural generation process adopted by the proposed framework:

```text
Synthetic Person
        ↓
Mobility Layer
(Trip Purpose)
        ↓
Temporal Layer
(Trip Start Time)
        ↓
Travel Mode Layer
        ↓
POI Category Determination
        ↓
Opportunity Layer
        ↓
Destination Selection
```

In contrast, **NTS0502a** is designed for situations where the trip start time is already known and the objective is to analyse the distribution of trip purposes within each time interval. This represents a reverse inference process that is inconsistent with the generative workflow proposed in this study.

Accordingly, **NTS0502b** was selected to ensure that the Temporal Layer functions as a behavioural generation component, maintaining consistency with the overall layer-based mobility modelling framework.

In [1]:
import os
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [3]:
nts0502b = pd.read_excel("../Dataset(mobility)/nts0502-Trip start time by trip purpose (Monday to Friday only) England, 2002 onwards.ods", sheet_name='NTS0502b_purpose_by_start_time', skiprows=5)

In [4]:
display(nts0502b.head())

,Year [note 2],Trip purpose,0000 to 0059 (%),0100 to 0159 (%),0200 to 0259 (%),0300 to 0359 (%),0400 to 0459 (%),0500 to 0559 (%),0600 to 0659 (%),0700 to 0759 (%),0800 to 0859 (%),0900 to 0959 (%),1000 to 1059 (%),1100 to 1159 (%),1200 to 1259 (%),1300 to 1359 (%),1400 to 1459 (%),1500 to 1559 (%),1600 to 1659 (%),1700 to 1759 (%),1800 to 1859 (%),1900 to 1959 (%),2000 to 2059 (%),2100 to 2159 (%),2200 to 2259 (%),2300 to 2359 (%),All day (%),Unweighted sample size: trips (thousands) (number)
0,2002 to 2006,Commuting,0.205698,0.101931,0.104566,0.182574,0.611806,2.449245,6.286387,15.986476,15.087049,3.656802,1.434472,1.576268,2.830811,3.998375,3.266973,4.087944,9.286012,15.321603,6.496979,2.288845,1.582217,1.308415,1.182520,0.666031,100,207.075
1,2002 to 2006,Business,0.113469,0.076103,0.047455,0.036680,0.139022,0.742937,2.643129,7.036194,10.194576,7.817521,6.798939,7.197912,7.581983,7.629309,6.934835,7.728231,9.433110,7.337461,4.259686,2.396614,1.487375,1.135503,0.802173,0.429784,100,48.075
2,2002 to 2006,Education,0.004818,0.000000,0.001001,0.003418,0.002020,0.006047,0.082235,6.011576,41.064773,2.343561,1.312823,1.452119,2.073874,1.840750,2.279897,32.743377,4.877663,2.249846,0.765730,0.279634,0.289271,0.227910,0.058754,0.028904,100,99.820
3,2002 to 2006,Escort education,0.001225,0.000000,0.004095,0.028255,0.002498,0.005781,0.045425,2.077776,35.409744,7.974474,0.520823,2.095834,1.765297,0.923104,7.652706,35.109558,3.713478,1.677470,0.426319,0.181188,0.221839,0.130278,0.025667,0.007166,100,75.727
4,2002 to 2006,Shopping,0.023158,0.003951,0.000420,0.003484,0.005803,0.043154,0.201715,0.966488,2.895945,8.350572,12.269701,12.792924,10.395929,9.251785,9.247824,8.733509,7.756958,5.720232,4.792887,3.372280,1.852103,0.890751,0.307148,0.121280,100,202.265


In [5]:
nts0502b.columns

Index(['Year [note 2]', 'Trip purpose', '0000 to 0059 (%)', '0100 to 0159 (%)',
       '0200 to 0259 (%)', '0300 to 0359 (%)', '0400 to 0459 (%)',
       '0500 to 0559 (%)', '0600 to 0659 (%)', '0700 to 0759 (%)',
       '0800 to 0859 (%)', '0900 to 0959 (%)', '1000 to 1059 (%)',
       '1100 to 1159 (%)', '1200 to 1259 (%)', '1300 to 1359 (%)',
       '1400 to 1459 (%)', '1500 to 1559 (%)', '1600 to 1659 (%)',
       '1700 to 1759 (%)', '1800 to 1859 (%)', '1900 to 1959 (%)',
       '2000 to 2059 (%)', '2100 to 2159 (%)', '2200 to 2259 (%)',
       '2300 to 2359 (%)', 'All day (%)',
       'Unweighted sample size: trips (thousands) (number)'],
      dtype='object')

In [6]:
nts0502b['Year [note 2]'].unique()

array(['2002 to 2006', '2003 to 2007', '2004 to 2008', '2005 to 2009',
       '2006 to 2010', '2007 to 2011', '2008 to 2012', '2009 to 2013',
       '2010 to 2014', '2011 to 2015', '2012 to 2016', '2013 to 2017',
       '2014 to 2018', '2015 to 2019', 2020, 2021, 2022, '2023 to 2024'],
      dtype=object)

Step 2: Create a copy and preserve the raw dataset

In [7]:
nts0502b_processed = nts0502b.copy()
nts0502b_processed = nts0502b_processed[
    nts0502b_processed["Year [note 2]"] == "2023 to 2024"].copy()

Step 3: Filter the DataFrame to include only the selected columns

In [8]:
import numpy as np

# Working copy
nts0502b_processed = nts0502b.copy()

# Select latest period
nts0502b_processed = nts0502b_processed[
    nts0502b_processed["Year [note 2]"] == "2023 to 2024"
].copy()

# Columns to keep
base_columns = ["Year [note 2]", "Trip purpose"]

hourly_time_columns = [
    "0000 to 0059 (%)",
    "0100 to 0159 (%)",
    "0200 to 0259 (%)",
    "0300 to 0359 (%)",
    "0400 to 0459 (%)",
    "0500 to 0559 (%)",
    "0600 to 0659 (%)",
    "0700 to 0759 (%)",
    "0800 to 0859 (%)",
    "0900 to 0959 (%)",
    "1000 to 1059 (%)",
    "1100 to 1159 (%)",
    "1200 to 1259 (%)",
    "1300 to 1359 (%)",
    "1400 to 1459 (%)",
    "1500 to 1559 (%)",
    "1600 to 1659 (%)",
    "1700 to 1759 (%)",
    "1800 to 1859 (%)",
    "1900 to 1959 (%)",
    "2000 to 2059 (%)",
    "2100 to 2159 (%)",
    "2200 to 2259 (%)",
    "2300 to 2359 (%)",
    "All day (%)"
]

nts0502b_processed = nts0502b_processed[base_columns + hourly_time_columns]

Step 4: Take a look

In [9]:
nts0502b_processed.head()

,Year [note 2],Trip purpose,0000 to 0059 (%),0100 to 0159 (%),0200 to 0259 (%),0300 to 0359 (%),0400 to 0459 (%),0500 to 0559 (%),0600 to 0659 (%),0700 to 0759 (%),0800 to 0859 (%),0900 to 0959 (%),1000 to 1059 (%),1100 to 1159 (%),1200 to 1259 (%),1300 to 1359 (%),1400 to 1459 (%),1500 to 1559 (%),1600 to 1659 (%),1700 to 1759 (%),1800 to 1859 (%),1900 to 1959 (%),2000 to 2059 (%),2100 to 2159 (%),2200 to 2259 (%),2300 to 2359 (%),All day (%)
153,2023 to 2024,Commuting,0.239840,0.148222,0.111912,0.218639,0.827896,3.058740,7.965453,15.264471,13.328675,3.658019,1.825335,1.631218,2.191046,3.136569,3.486577,5.383947,10.382031,13.261369,6.275847,2.542718,1.758006,1.131909,1.399447,0.772115,100
154,2023 to 2024,Business,0.041416,0.015177,0.060895,0.081876,0.220996,1.074968,4.589156,8.696725,11.267973,7.465897,6.009973,7.555960,6.552888,6.630029,6.343139,7.162665,9.123441,7.463486,3.991702,1.922051,1.518446,1.129986,0.698915,0.382243,100
155,2023 to 2024,Education,0.000000,0.000000,0.000000,0.030642,0.004397,0.011165,0.253959,9.799807,37.755919,1.758576,0.968553,1.160566,1.471589,0.844463,2.796316,33.281601,5.483792,2.674043,0.772700,0.471058,0.207967,0.186490,0.049809,0.016588,100
156,2023 to 2024,Escort education,0.000000,0.000000,0.007056,0.012924,0.004907,0.022035,0.181196,5.149004,38.714753,3.408739,0.375344,1.210844,1.091111,0.519388,9.301491,30.963633,5.211611,2.884029,0.528071,0.260502,0.096282,0.020906,0.030402,0.005771,100
157,2023 to 2024,Shopping,0.016551,0.002386,0.000000,0.000000,0.001099,0.047941,0.335906,1.288284,3.181589,7.245089,11.792016,12.445005,11.116141,9.373412,8.713807,7.572346,7.269566,6.474185,5.717458,4.203831,2.133739,0.761697,0.220305,0.087645,100


Step 5: Check whether there are missing values

In [10]:
nts0502b_processed.isna().sum()

Year [note 2]       0
Trip purpose        0
0000 to 0059 (%)    0
0100 to 0159 (%)    0
0200 to 0259 (%)    0
0300 to 0359 (%)    0
0400 to 0459 (%)    0
0500 to 0559 (%)    0
0600 to 0659 (%)    0
0700 to 0759 (%)    0
0800 to 0859 (%)    0
0900 to 0959 (%)    0
1000 to 1059 (%)    0
1100 to 1159 (%)    0
1200 to 1259 (%)    0
1300 to 1359 (%)    0
1400 to 1459 (%)    0
1500 to 1559 (%)    0
1600 to 1659 (%)    0
1700 to 1759 (%)    0
1800 to 1859 (%)    0
1900 to 1959 (%)    0
2000 to 2059 (%)    0
2100 to 2159 (%)    0
2200 to 2259 (%)    0
2300 to 2359 (%)    0
All day (%)         0
dtype: int64

In [11]:
nts0502b_processed.describe()

,0000 to 0059 (%),0100 to 0159 (%),0200 to 0259 (%),0300 to 0359 (%),0400 to 0459 (%),0500 to 0559 (%),0600 to 0659 (%),0700 to 0759 (%),0800 to 0859 (%),0900 to 0959 (%),1000 to 1059 (%),1100 to 1159 (%),1200 to 1259 (%),1300 to 1359 (%),1400 to 1459 (%),1500 to 1559 (%),1600 to 1659 (%),1700 to 1759 (%),1800 to 1859 (%),1900 to 1959 (%),2000 to 2059 (%),2100 to 2159 (%),2200 to 2259 (%),2300 to 2359 (%),All day (%)
count,9.000000,9.000000,9.000000,9.000000,9.000000,9.000000,9.000000,9.000000,9.000000,9.000000,9.000000,9.000000,9.000000,9.000000,9.000000,9.000000,9.000000,9.000000,9.000000,9.000000,9.000000,9.000000,9.000000,9.000000,9.0
mean,0.065198,0.031715,0.032137,0.056023,0.173117,0.685426,2.123417,6.290868,15.022612,5.445541,5.477453,5.813655,5.622714,5.167421,6.320031,13.256032,7.709237,7.292082,5.152196,3.534515,2.090421,1.293560,0.890673,0.453955,100.0
std,0.075929,0.046719,0.035439,0.066935,0.259898,0.962291,2.599644,4.383534,13.686932,2.174203,3.824414,3.822052,3.368214,3.021183,2.118041,10.836143,1.671373,3.228213,3.129330,2.652195,1.596922,1.293130,0.895210,0.503773,0.0
min,0.000000,0.000000,0.000000,0.000000,0.001099,0.011165,0.181196,1.288284,2.790124,1.758576,0.375344,1.160566,1.091111,0.519388,2.796316,5.383947,5.211611,2.674043,0.528071,0.260502,0.096282,0.020906,0.030402,0.005771,100.0
25%,0.016551,0.002386,0.007056,0.012924,0.004907,0.047941,0.335906,4.342872,6.379562,3.658019,1.825335,1.631218,2.191046,3.136569,6.070037,7.234098,7.269566,6.474185,3.991702,1.922051,1.518446,0.761697,0.220305,0.087645,100.0
50%,0.041416,0.015177,0.024006,0.031787,0.090394,0.284516,0.916091,5.149004,11.267973,5.540024,6.001584,6.306697,6.552888,6.220854,6.343139,7.763935,7.599805,7.463486,5.920261,3.930434,2.133739,1.131909,0.755093,0.368079,100.0
75%,0.076578,0.037278,0.031837,0.066541,0.182320,0.694165,2.137848,8.696725,13.328675,7.409701,7.676569,7.555960,7.007568,7.015506,7.071918,11.611308,9.020227,8.580281,6.275847,4.203831,2.449824,1.262759,1.049850,0.539076,100.0
max,0.239840,0.148222,0.111912,0.218639,0.827896,3.058740,7.965453,15.264471,38.714753,7.738354,11.792016,12.445005,11.116141,9.373412,9.301491,33.281601,10.382031,13.261369,10.814387,8.792454,5.524778,4.475472,2.934002,1.619149,100.0


In [12]:
nts0502b_processed

,Year [note 2],Trip purpose,0000 to 0059 (%),0100 to 0159 (%),0200 to 0259 (%),0300 to 0359 (%),0400 to 0459 (%),0500 to 0559 (%),0600 to 0659 (%),0700 to 0759 (%),0800 to 0859 (%),0900 to 0959 (%),1000 to 1059 (%),1100 to 1159 (%),1200 to 1259 (%),1300 to 1359 (%),1400 to 1459 (%),1500 to 1559 (%),1600 to 1659 (%),1700 to 1759 (%),1800 to 1859 (%),1900 to 1959 (%),2000 to 2059 (%),2100 to 2159 (%),2200 to 2259 (%),2300 to 2359 (%),All day (%)
153,2023 to 2024,Commuting,0.239840,0.148222,0.111912,0.218639,0.827896,3.058740,7.965453,15.264471,13.328675,3.658019,1.825335,1.631218,2.191046,3.136569,3.486577,5.383947,10.382031,13.261369,6.275847,2.542718,1.758006,1.131909,1.399447,0.772115,100
154,2023 to 2024,Business,0.041416,0.015177,0.060895,0.081876,0.220996,1.074968,4.589156,8.696725,11.267973,7.465897,6.009973,7.555960,6.552888,6.630029,6.343139,7.162665,9.123441,7.463486,3.991702,1.922051,1.518446,1.129986,0.698915,0.382243,100
155,2023 to 2024,Education,0.000000,0.000000,0.000000,0.030642,0.004397,0.011165,0.253959,9.799807,37.755919,1.758576,0.968553,1.160566,1.471589,0.844463,2.796316,33.281601,5.483792,2.674043,0.772700,0.471058,0.207967,0.186490,0.049809,0.016588,100
156,2023 to 2024,Escort education,0.000000,0.000000,0.007056,0.012924,0.004907,0.022035,0.181196,5.149004,38.714753,3.408739,0.375344,1.210844,1.091111,0.519388,9.301491,30.963633,5.211611,2.884029,0.528071,0.260502,0.096282,0.020906,0.030402,0.005771,100
157,2023 to 2024,Shopping,0.016551,0.002386,0.000000,0.000000,0.001099,0.047941,0.335906,1.288284,3.181589,7.245089,11.792016,12.445005,11.116141,9.373412,8.713807,7.572346,7.269566,6.474185,5.717458,4.203831,2.133739,0.761697,0.220305,0.087645,100
158,2023 to 2024,"Other work, other escort and personal business",0.068853,0.023436,0.022414,0.007135,0.090394,0.283279,0.916091,4.342872,9.326097,7.409701,7.676569,7.461490,8.141917,7.015506,6.762664,8.330755,9.020227,8.580281,5.941158,3.930434,2.372549,1.153004,0.755093,0.368079,100
159,2023 to 2024,"Visiting friends, entertainment and sport",0.116263,0.047257,0.031109,0.031787,0.046516,0.284516,0.738222,1.559782,2.790124,4.785472,5.707837,6.306697,6.911001,6.220854,6.070037,7.234098,7.599805,9.358378,10.814387,8.792454,5.524778,4.475472,2.934002,1.619149,100
160,2023 to 2024,"Holiday, day trip and other",0.027284,0.011680,0.024006,0.066541,0.182320,0.694165,2.137848,4.780326,6.379562,7.738354,8.939868,8.345975,7.007568,7.187170,7.071918,7.763935,7.444504,7.083675,6.408183,5.516998,2.752198,1.262759,0.878233,0.294930,100
161,2023 to 2024,All purposes,0.076578,0.037278,0.031837,0.054663,0.179531,0.692024,1.992918,5.736538,12.458817,5.540024,6.001584,6.205140,6.121168,5.579396,6.334324,11.611308,7.848156,7.849292,5.920261,4.170592,2.449824,1.519820,1.049850,0.539076,100


### Conversion from Percentage to Probability

The NTS0502b dataset reports temporal distributions as **percentages**, where the hourly probabilities associated with each trip purpose sum to 100%. Verification confirmed that each trip purpose forms a valid probability distribution across the 24 hourly departure intervals.

Although these values already represent probabilities conceptually, they are expressed on a percentage scale (0–100). To ensure consistency with the probabilistic representations used throughout the proposed framework, all percentage values were converted to decimal probabilities by dividing each value by 100.

This conversion does not modify the underlying probability distribution; it simply changes its numerical representation from percentages to probabilities within the interval \([0,1]\). Using decimal probabilities provides a consistent format for subsequent probabilistic sampling and integration with the other behavioural components of the framework.

In [13]:
temporal_probability_matrix = nts0502b_processed.copy()

temporal_probability_matrix[hourly_time_columns] = (
    temporal_probability_matrix[hourly_time_columns] / 100
)

display(temporal_probability_matrix)

,Year [note 2],Trip purpose,0000 to 0059 (%),0100 to 0159 (%),0200 to 0259 (%),0300 to 0359 (%),0400 to 0459 (%),0500 to 0559 (%),0600 to 0659 (%),0700 to 0759 (%),0800 to 0859 (%),0900 to 0959 (%),1000 to 1059 (%),1100 to 1159 (%),1200 to 1259 (%),1300 to 1359 (%),1400 to 1459 (%),1500 to 1559 (%),1600 to 1659 (%),1700 to 1759 (%),1800 to 1859 (%),1900 to 1959 (%),2000 to 2059 (%),2100 to 2159 (%),2200 to 2259 (%),2300 to 2359 (%),All day (%)
153,2023 to 2024,Commuting,0.002398,0.001482,0.001119,0.002186,0.008279,0.030587,0.079655,0.152645,0.133287,0.036580,0.018253,0.016312,0.021910,0.031366,0.034866,0.053839,0.103820,0.132614,0.062758,0.025427,0.017580,0.011319,0.013994,0.007721,1.0
154,2023 to 2024,Business,0.000414,0.000152,0.000609,0.000819,0.002210,0.010750,0.045892,0.086967,0.112680,0.074659,0.060100,0.075560,0.065529,0.066300,0.063431,0.071627,0.091234,0.074635,0.039917,0.019221,0.015184,0.011300,0.006989,0.003822,1.0
155,2023 to 2024,Education,0.000000,0.000000,0.000000,0.000306,0.000044,0.000112,0.002540,0.097998,0.377559,0.017586,0.009686,0.011606,0.014716,0.008445,0.027963,0.332816,0.054838,0.026740,0.007727,0.004711,0.002080,0.001865,0.000498,0.000166,1.0
156,2023 to 2024,Escort education,0.000000,0.000000,0.000071,0.000129,0.000049,0.000220,0.001812,0.051490,0.387148,0.034087,0.003753,0.012108,0.010911,0.005194,0.093015,0.309636,0.052116,0.028840,0.005281,0.002605,0.000963,0.000209,0.000304,0.000058,1.0
157,2023 to 2024,Shopping,0.000166,0.000024,0.000000,0.000000,0.000011,0.000479,0.003359,0.012883,0.031816,0.072451,0.117920,0.124450,0.111161,0.093734,0.087138,0.075723,0.072696,0.064742,0.057175,0.042038,0.021337,0.007617,0.002203,0.000876,1.0
158,2023 to 2024,"Other work, other escort and personal business",0.000689,0.000234,0.000224,0.000071,0.000904,0.002833,0.009161,0.043429,0.093261,0.074097,0.076766,0.074615,0.081419,0.070155,0.067627,0.083308,0.090202,0.085803,0.059412,0.039304,0.023725,0.011530,0.007551,0.003681,1.0
159,2023 to 2024,"Visiting friends, entertainment and sport",0.001163,0.000473,0.000311,0.000318,0.000465,0.002845,0.007382,0.015598,0.027901,0.047855,0.057078,0.063067,0.069110,0.062209,0.060700,0.072341,0.075998,0.093584,0.108144,0.087925,0.055248,0.044755,0.029340,0.016191,1.0
160,2023 to 2024,"Holiday, day trip and other",0.000273,0.000117,0.000240,0.000665,0.001823,0.006942,0.021378,0.047803,0.063796,0.077384,0.089399,0.083460,0.070076,0.071872,0.070719,0.077639,0.074445,0.070837,0.064082,0.055170,0.027522,0.012628,0.008782,0.002949,1.0
161,2023 to 2024,All purposes,0.000766,0.000373,0.000318,0.000547,0.001795,0.006920,0.019929,0.057365,0.124588,0.055400,0.060016,0.062051,0.061212,0.055794,0.063343,0.116113,0.078482,0.078493,0.059203,0.041706,0.024498,0.015198,0.010498,0.005391,1.0


### Probability Verification

After converting percentages to decimal probabilities, each trip purpose was verified to ensure that the probabilities across all 24 hourly departure intervals summed to one. This confirms that each row represents a valid conditional probability distribution and can therefore be directly used for probabilistic sampling within the Temporal Layer.